In [1]:
import os
import sys
import random
import functools
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RepeatedKFold
import warnings
warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '3407'
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

def set_all_random_seeds(seed=3407):
    np.random.seed(seed)
    random.seed(seed)
    print(f"已设置随机种子为: {seed}")

def print_data_shapes(X_train, y_train, X_test, y_test):
    print(f"训练集形状: {X_train.shape}")
    print(f"测试集形状: {X_test.shape}")

def objective(trial, X_train, y_train, cv_folds, seed=3407):
 
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': seed,
        'n_jobs': 1,
        'verbose': 0
    }
   
    model = RandomForestRegressor(**params)
    
    rkf = RepeatedKFold(n_splits=cv_folds, n_repeats=5, random_state=seed)
    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=rkf, scoring='neg_mean_squared_error',
        n_jobs=1
    )
    mse_scores = -cv_scores
    avg_mse = np.mean(mse_scores)
    mse_std = np.std(mse_scores)
    stability_penalty = 0.1 * mse_std / avg_mse if avg_mse > 0 else 0
    return avg_mse + stability_penalty

def optimize_hyperparameters(X_train, y_train, n_trials=100, cv_folds=10, seed=3407):
    print("开始 Optuna 超参数优化（随机森林）...")

    sampler = TPESampler(
        seed=seed,
        n_startup_trials=10,
        group=False,
        warn_independent_sampling=False
    )

    study = optuna.create_study(
        direction='minimize',
        study_name=f'random_forest_regression_seed{seed}',
        storage=None,
        load_if_exists=False,
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
    )
    
    objective_partial = functools.partial(
        objective,
        X_train=X_train,
        y_train=y_train,
        cv_folds=cv_folds,
        seed=seed
    )
    
    study.optimize(objective_partial, n_trials=n_trials, n_jobs=1, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"\n最佳超参数: {best_params}")
    print(f"最佳交叉验证 MSE: {study.best_value:.4f}")
    
    best_params['random_state'] = seed
    return best_params, study

def train_best_model(X_train, y_train, best_params, seed=3407):
   
    final_params = best_params.copy()
    final_params.update({
        'random_state': seed,
        'n_jobs': 1,
        'verbose': 0
    })
    model = RandomForestRegressor(**final_params)
    model.fit(X_train, y_train)
    return model, (X_train, y_train)

def evaluate_model(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    mse_diff = abs(train_mse - test_mse) / train_mse * 100 if train_mse > 0 else 0
    r2_diff = abs(train_r2 - test_r2) * 100
    
    print("\n" + "="*50)
    print("模型性能评估")
    print("="*50)
    print(f"{'指标':<15} {'训练集':<15} {'测试集':<15} {'差异(%)':<10}")
    print("-"*50)
    print(f"{'MSE':<15} {train_mse:<15.4f} {test_mse:<15.4f} {mse_diff:<10.2f}")
    print(f"{'RMSE':<15} {np.sqrt(train_mse):<15.4f} {np.sqrt(test_mse):<15.4f} {'-':<10}")
    print(f"{'MAE':<15} {train_mae:<15.4f} {test_mae:<15.4f} {'-':<10}")
    print(f"{'R²':<15} {train_r2:<15.4f} {test_r2:<15.4f} {r2_diff:<10.2f}")
    print("="*50)
    
    metrics_dict = {
        'train_mse': train_mse, 'test_mse': test_mse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'train_r2': train_r2, 'test_r2': test_r2,
        'mse_diff_percent': mse_diff, 'r2_diff_percent': r2_diff
    }
    predictions = {'y_train_pred': y_train_pred, 'y_test_pred': y_test_pred}
    return metrics_dict, predictions

def save_model(model, filepath='best_random_forest_model.pkl'):
    import joblib
    joblib.dump(model, filepath)
    print(f"模型已保存到: {filepath}")

def random_forest_optimization_pipeline(X_train, y_train, X_test, y_test, n_trials=100, cv_folds=5, seed=3407):
    set_all_random_seeds(seed)
    print_data_shapes(X_train, y_train, X_test, y_test)
    
    best_params, study = optimize_hyperparameters(
        X_train, y_train, n_trials, cv_folds, seed
    )
    model, train_data = train_best_model(X_train, y_train, best_params, seed)
    metrics, predictions = evaluate_model(model, X_train, y_train, X_test, y_test)
    save_model(model, 'RandomForest_regression_model.pkl')
    print("\n优化完成！")
    return model, best_params, metrics, study, predictions, train_data

def main():
    SEED = 36
    
    X_train = pd.read_csv("x_train.csv")
    X_test  = pd.read_csv("x_test.csv")
    y_train = pd.read_csv("y_train.csv")
    y_test  = pd.read_csv("y_test.csv")
    
    if len(y_train.shape) > 1 and y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    if len(y_test.shape) > 1 and y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]
    
    model, best_params, metrics, study, predictions, train_data = random_forest_optimization_pipeline(
        X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
        n_trials=100,
        cv_folds=10,
        seed=SEED
    )
    
    print(f"\n最佳参数数量: {len(best_params)}")
    print(f"训练集 R²: {metrics['train_r2']:.4f}")
    print(f"测试集 R²: {metrics['test_r2']:.4f}")
    
    results = {
        'model': model, 'best_params': best_params, 'metrics': metrics,
        'study': study, 'predictions': predictions, 'train_data': train_data
    }
    return results

if __name__ == "__main__":
    results = main()

[I 2026-04-09 09:23:28,590] A new study created in memory with name: random_forest_regression_seed36


已设置随机种子为: 36
训练集形状: (921, 30)
测试集形状: (231, 30)
开始 Optuna 超参数优化（随机森林）...


Best trial: 0. Best value: 14.3144:   1%|          | 1/100 [00:16<27:39, 16.77s/it]

[I 2026-04-09 09:23:45,355] Trial 0 finished with value: 14.314430512521868 and parameters: {'n_estimators': 378, 'max_depth': 13, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 14.314430512521868.


Best trial: 0. Best value: 14.3144:   2%|▏         | 2/100 [00:19<13:53,  8.51s/it]

...

[I 2026-04-09 10:33:27,276] Trial 97 finished with value: 11.79142140048748 and parameters: {'n_estimators': 169, 'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'bootstrap': True}. Best is trial 87 with value: 11.772187978623496.


Best trial: 87. Best value: 11.7722:  99%|█████████▉| 99/100 [1:10:23<00:21, 21.51s/it]

[I 2026-04-09 10:33:51,988] Trial 98 finished with value: 12.120087480082285 and parameters: {'n_estimators': 162, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True}. Best is trial 87 with value: 11.772187978623496.


Best trial: 87. Best value: 11.7722: 100%|██████████| 100/100 [1:10:37<00:00, 42.38s/it]


[I 2026-04-09 10:34:06,354] Trial 99 finished with value: 11.801799414726423 and parameters: {'n_estimators': 87, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': None, 'bootstrap': True}. Best is trial 87 with value: 11.772187978623496.

最佳超参数: {'n_estimators': 78, 'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'bootstrap': True}
最佳交叉验证 MSE: 11.7722

模型性能评估
指标              训练集             测试集             差异(%)     
--------------------------------------------------
MSE             1.6729          19.4063         1060.04   
RMSE            1.2934          4.4053          -         
MAE             0.8424          2.6505          -         
R²              0.9791          0.7591          22.01     
模型已保存到: RandomForest_regression_model.pkl

优化完成！

最佳参数数量: 7
训练集 R²: 0.9791
测试集 R²: 0.7591
